# Lab: Relationships in Space and Time

Select **two distinct datasets that each contain both spatial and temporal information**. Ask **one research question for each dataset** and answer it using the six-step reproducible analysis workflow. Create **two different types of Plotly Express figures for each question**, for a total of two research questions and four figures.

Together, each pair of figures must help answer where and when the pattern occurs. A map and a time-series plot are one useful combination; other combinations are welcome when you explain how they address both space and time.

## Learning goals

- Select and document datasets with usable location and time features.
- Ask focused questions that connect a measurement or event count to both space and time.
- Prepare dates, geographic identifiers, and summaries appropriately for the question.
- Create and compare two complementary interactive figures for each question.
- Check plotted values and ordering against the records.
- Support conclusions with evidence and explain the limitations of the data and displays.

## The reproducible analysis workflow

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can help answer it?
3. **Evidence:** What should we calculate and display to answer the question?
4. **Check:** Does the result make sense in context and agree with the original records?
5. **Conclusion:** What claim is supported?
6. **Limitation:** What should we avoid concluding?

Complete all six steps separately for each dataset. Inspect missing values during Data; use Check to investigate the actual figures and calculations. Use separate code cells for individual operations and add cells as needed.

## Using AI tools responsibly

You may use AI to ask questions, debug code, or receive feedback. You remain responsible for understanding the code, reading the source documentation, and checking every claim against your output. Complete the AI-use reflection at the end.

## Select your two datasets

Each dataset needs:

- A documented publisher and feature definitions or data dictionary.
- A usable **time feature**, such as a timestamp, date, month, or year, with observations from multiple times.
- A usable **spatial feature**, such as coordinates, station locations, or named geographic areas with matching identifiers, covering multiple places or a moving object's locations.
- A quantitative measurement with known units, or event records that support a meaningful count.
- Enough usable observations to compare places and periods. A file with dates from one fixed location alone will not support a spatial comparison.

Use two distinct datasets, not two filters of the same file. Choose your own questions and records rather than reproducing a worked lecture example. The suggestions below are starting points, not required choices. You may select another documented source.

Choose manageable geographic and time ranges before downloading. A country or city name does not automatically provide coordinates; document any station lookup, geographic-code lookup, or boundary file you need. Save fixed copies in this notebook's `data` folder, preserve the original records, and use relative paths. Include supporting lookup files with your submission.

### Candidate datasets

| Candidate and source | Useful spatial and time information | Possible question and figure pair |
| --- | --- | --- |
| [USGS earthquake catalog](https://earthquake.usgs.gov/earthquakes/search/) and [CSV field definitions](https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php) | Event time, coordinates, magnitude, and depth. Limit the dates, geographic bounds, and magnitude range; download CSV. | Where did recorded earthquakes cluster during a selected period, and how did monthly counts change? Use a point or density map and a monthly count plot. |
| [NOAA daily weather observations](https://www.ncei.noaa.gov/cdo-web/search) and [GHCNd documentation](https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily) | Station identifiers, dates, temperature or precipitation, and documented station locations. Select several stations and the same time span. | How did monthly precipitation differ among selected locations during a year? Use a station map of monthly summaries and a line plot or location-by-month heatmap. |
| [EPA daily air-quality data](https://www.epa.gov/outdoor-air-quality-data/download-daily-data) and [file documentation](https://aqs.epa.gov/aqsweb/airdata/download_files.html) | Monitor or county identifiers, dates, pollutant measurements or AQI. Use one pollutant and compatible units; obtain station coordinates if making a point map. | Where and when were PM2.5 concentrations highest among selected monitors? Use a map and a time-series plot grouped by monitor. |
| [NOAA Storm Events Database](https://www.ncei.noaa.gov/access/storm-events-database/) | Event dates, states/counties, event types, and locations when reported. Start with one event type, a few states, and a limited period. | How did reported hail-event counts vary across selected counties and months? Use a county choropleth and a county-by-month heatmap. County boundaries/codes are needed for the map. |

Adapt the question to the fields actually available. Counts of reported events or monitor observations can reflect reporting and sampling effort as well as the underlying phenomenon. Do not treat missing observations as zero events or average measurements with incompatible units.

## Choose two complementary figures per question

Choose **different chart types**, not two copies that differ only in color, title, filter, or static-versus-animated display. You may reuse a useful chart type with the second dataset. An animation counts as one figure, regardless of its number of frames. Animation is optional; a well-chosen static comparison can be easier to interpret.

| Figure | Plotly Express tools | What it can help you see |
| --- | --- | --- |
| Time-series line plot | `px.line()` | How a measurement changes over time; color or separate panels can distinguish locations. Sort dates within each series before connecting points. |
| Point or bubble map | `px.scatter_geo()` or `px.scatter_map()` | Where observations occur; color and size can show quantitative values. Time can appear in hover information, filters, or animation. |
| Choropleth | `px.choropleth()` | How an area-level summary varies geographically, optionally animated by period. Match country/state codes or supply matching GeoJSON boundaries. |
| Bubble scatterplot | `px.scatter()` | How two quantitative features relate; geographic groups and time can be represented by color and animation. Explain how location is represented if this replaces a map. |
| Geographic path — an additional option | `px.line_geo()` or `px.line_map()` | A recorded route or moving object's sequence of positions. Sort within each tracked object; do not connect unrelated events into a path. |
| Spatial density map — an additional option | `px.density_map()` | Where many nearby point observations concentrate. The smoothing radius affects the appearance; this is not automatically a rate or a map of risk. |
| Location-by-time heatmap — an additional option | `px.imshow()` | Rows represent locations, columns represent ordered periods, and color represents a count or measurement. Summarize to one value per location-period first. |

Use the [Plotly map guide](https://plotly.com/python/map-configuration/), [heatmap guide](https://plotly.com/python/heatmaps/), and [animation guide](https://plotly.com/python/animations/) for examples and parameters. Tile maps need access to background tiles; geographic maps may also need internet access for boundary outlines.

### Starting patterns for additional chart types

These are **adaptation examples**, not code to run unchanged. Replace the DataFrame and feature names with yours. They do not add requirements to your four figures.

**Path of a tracked object:** only use this when consecutive rows describe the same object's movement.

```python
track_df.sort_values(['object_id', 'date'], inplace=True)
fig = px.line_geo(track_df, lat='latitude', lon='longitude',
                  color='object_id', hover_data=['date'],
                  title='Recorded paths')
fig.update_layout(height=500)
fig.show()
```

**Density of point observations:** omit `z` to give every row equal weight. Document any deliberate weighting.

```python
fig = px.density_map(events_df, lat='latitude', lon='longitude',
                     radius=15, map_style='open-street-map',
                     title='Concentration of recorded events')
fig.update_layout(height=500)
fig.show()
```

**Location-by-month heatmap:** this example uses a mean measurement. Use a sum or count only when that statistic answers your question. Missing cells remain missing unless you can establish that a zero is appropriate.

```python
observations_df['month'] = observations_df['date'].dt.to_period('M').astype(str)
monthly_df = observations_df.groupby(['location', 'month'])['value'].mean().reset_index()
grid = monthly_df.pivot(index='location', columns='month', values='value')
grid = grid.reindex(sorted(grid.columns), axis=1)
fig = px.imshow(grid, aspect='auto',
                labels={'x':'Month', 'y':'Location', 'color':'Mean value'},
                title='Mean measurement by location and month')
fig.update_layout(height=500)
fig.show()
```

### Make the comparison interpretable

Use descriptive titles, labels with units, and informative hover fields. Start with `fig.update_layout(height=500)` and adjust only if needed. Keep scales consistent when comparing frames or panels. Use a manageable number of time periods—monthly, quarterly, or yearly frames often work better than hundreds of daily frames. Explain any aggregation and retain the original dates.

For maps, confirm that geographic codes match the boundaries and that points appear in plausible places. Numeric identifiers are not quantitative measurements. A colored area or large marker does not necessarily mean greater risk: consider population, area, observation effort, and what the encoded value actually measures.

In [1]:
# Import Pandas and Plotly Express.
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Keep long text fields readable when tables are displayed.
pd.set_option("display.max_colwidth", 60)

# Store each figure in two formats so it displays in JupyterLab, VS Code, and the classic notebook.
pio.renderers.default = "plotly_mimetype+notebook_connected"


## Investigation 1: Your dataset and research question

### Step 1: Question

Ask one focused question that depends on **both where and when** something occurs. Name the measurement or count, the places, and the period. Record a prediction before producing the figures.

**What is your research question, and what do you predict? Why is it useful to consider both space and time?**

Your answer:

**Question.** Where in the United States has the IT power capacity of large AI data centers been built, and when did each state's capacity come online, between the first quarter of 2023 and the third quarter of 2026? The measurement is estimated IT power in megawatts (MW) at the sites Epoch AI tracks, summed by state at each quarter end.

**Prediction.** I expect the capacity to be concentrated in a few states, with Texas and Virginia at the top. Texas has the land and power for very large sites, and northern Virginia is the best-known data center region in the country. I expect most of the capacity to have come online in 2025 and 2026, because sites this large take a long time to build and the push to build them is recent.

**Why space and time together.** A total by state hides when the building happened, and a national timeline hides where it happened. Electricity is supplied regionally, so the same number of megawatts means something different if it arrives in one state within a year than if it is spread across many states over several years.


### Step 2: Data

Document the source and explain what one row represents. Load a fixed copy, inspect the first and last rows and shape, use `.info()` to inspect the data storage types and non-missing values, and count missing values. Parse dates when needed and preserve geographic identifiers appropriately.

**Give the dataset title, publisher, direct source URL, dictionary reference, and access date. Identify your saved file and any lookup or boundary files.**

Your answer:

- **Title:** AI data centers
- **Publisher:** Epoch AI
- **Direct source URL:** https://epoch.ai/data/data_centers/data_centers.zip (dataset page: https://epoch.ai/data/ai-data-centers)
- **Dictionary reference:** https://epoch.ai/data/data-centers-documentation/records
- **Access date:** November 12, 2026
- **License and citation:** Creative Commons Attribution. Epoch AI, 'AI data centers'. Published online at epoch.ai. Retrieved from https://epoch.ai/data/ai-data-centers.

**Saved files.** `data/epoch_ai_data_centers.csv` is the download's `data_centers.csv` and `data/epoch_ai_data_center_timelines.csv` is its `data_center_timelines.csv`. Both are unchanged apart from the file names.

**Lookup file.** `data/us_state_codes.csv` holds the 50 state names and their two-letter codes. It was created for this lab from the U.S. Postal Service list of state abbreviations (Publication 28, Appendix B, https://pe.usps.com/text/pub28/28apb.htm). No boundary file is needed because Plotly Express draws state outlines from two-letter codes with `locationmode="USA-states"`.


**What does one row represent? Which places and dates are covered, and how were the observations collected?**

Your answer:

The dataset has two tables. In the site table, one row is one AI data center site. There are 96 sites, 77 in the United States and 19 in 11 other countries. In the timeline table, one row is one dated status record for one site, giving the site's estimated IT power and other measures as of that date. There are 575 records for the same 96 sites, dated from December 18, 2018 to January 1, 2030.

Epoch AI's documentation says the records are built from high-resolution satellite imagery, permits, and public documents, so the power values are estimates and not readings from the sites. Records dated after the access date are projections of planned construction. The dataset covers the large AI data centers Epoch AI tracks. It is not a list of every data center.


In [2]:
# Read dataset 1 from a relative path into a descriptive DataFrame ending in _df.
# Parse the time feature as appropriate for its documented format.
data_centers_df = pd.read_csv("data/epoch_ai_data_centers.csv")
timelines_df = pd.read_csv("data/epoch_ai_data_center_timelines.csv", parse_dates=["Date"])

# Supporting lookup: the 50 state names and their two-letter postal codes.
state_codes_df = pd.read_csv("data/us_state_codes.csv")


In [3]:
# Display .head(), .tail(), and .shape in separate cells. Add cells as needed.
data_centers_df.head()


,Name,Current H100 equivalents,Current power (MW),Current total capital cost (2025 USD billions),Owner,Users,Selected Sources,Calculations sheet,Project,Current chip types,All chip types,Investors,Construction companies,Energy companies,Country,Address
0,Colossus 2,1.111673e+06,946.0,35.836372,SpaceXAI #confident,"Anthropic #confident, Cursor #confident, SpaceXAI #confi...",- [WSJ profile of the Colossus data centers](https://www...,https://docs.google.com/spreadsheets/d/1oZpZ9nTTV6wW0xjc...,NaN,"B200,B300","B200,B300",NaN,NaN,NaN,United States,"5420 Tulane Rd, Memphis, TN 38109"
1,OpenAI Stargate Abilene,1.018696e+06,824.0,31.214768,Oracle #confident,OpenAI #confident,"- [Crusoe 2024 Impact Report, with details on Abilene](h...",https://docs.google.com/spreadsheets/d/1M5VPpNDGTPgssU1x...,Stargate #confident,"B200,B300","B200,B300","Softbank,OpenAI,Oracle",Mortenson,American Electric Power,United States,"5502 Spinks Rd, Abilene, TX 79601"
2,Anthropic-Amazon New Carlisle,8.184285e+05,1037.0,39.283634,Amazon #confident,Anthropic #confident,- [Air Construction Permit application for the northern ...,https://docs.google.com/spreadsheets/d/1umU-KLAB_6A-lKRC...,Project Rainier #confident,Trainium2,Trainium2,NaN,NaN,NaN,United States,"55001 Larrison Blvd, New Carlisle, IN 46552"
3,Microsoft Fairwater Atlanta,7.687687e+05,636.0,24.092952,Microsoft #confident,"OpenAI #likely, Microsoft #likely",- [2023 Air Permit application](https://drive.google.com...,https://docs.google.com/spreadsheets/d/1XeruHu2ES3UsSrsA...,Microsoft AI WAN #confident,B200,B200,NaN,NaN,Georgia Power,United States,"1435 Hwy 54 W, Fayetteville, GA 30214"
4,Google Pryor (North),6.366852e+05,368.0,13.940576,Google #confident,Google DeepMind #speculative,- [Google announces expansion of Pryor facility](https:/...,https://docs.google.com/spreadsheets/d/1lD_rxcr7NwyAXlJf...,NaN,"TPU v5p,TPU v5e","TPU v5p,TPU v5e",NaN,NaN,NaN,United States,"4581 Webb St, Pryor, OK 74361"


In [4]:
# Display the last rows of the site table.
data_centers_df.tail()


,Name,Current H100 equivalents,Current power (MW),Current total capital cost (2025 USD billions),Owner,Users,Selected Sources,Calculations sheet,Project,Current chip types,All chip types,Investors,Construction companies,Energy companies,Country,Address
91,OpenAI Stargate Shackelford,0.0,0.0,0.0,Oracle #confident,OpenAI #confident,"- [Vantage data center page with power capacity, number ...",https://docs.google.com/spreadsheets/d/1MzNM0bbxwhChbKF-...,Stargate #confident,NaN,Rubin,"Oracle,OpenAI",NaN,NaN,United States,"175 Private Road 1604, Abilene, TX 79601, Shackelford Co..."
92,OpenAI Stargate UAE,0.0,0.0,0.0,G42 #likely,OpenAI #confident,- [News update with timelapse of construction](https://w...,https://docs.google.com/spreadsheets/d/11es1WqQ3StAuMjFO...,Stargate #confident,NaN,"B300,Rubin","Softbank,Oracle,NVIDIA,Cisco,G42",NaN,NaN,United Arab Emirates,"Nexus L&T Project Office, Al Bihouth, Al Dhafrah, Abu Dh..."
93,OpenAI Stargate Wisconsin,0.0,0.0,0.0,Oracle #confident,OpenAI #confident,- [Data center page on Vantage website](https://vantage-...,https://docs.google.com/spreadsheets/d/1fjTxXhhaiFa7x1YB...,Stargate #confident,NaN,NaN,NaN,NaN,NaN,United States,NaN
94,QTS Cedar Rapids,0.0,0.0,0.0,Meta #speculative,Meta #speculative,- [MISO new load announcements](https://cdn.misoenergy.o...,https://docs.google.com/spreadsheets/d/1zXqoogyqGbmgvdu4...,NaN,NaN,"B300,Rubin",NaN,NaN,Alliant Energy,United States,"6200 76th Ave SW, Fairfax, IA 52228"
95,QTS Eagle Mountain,0.0,0.0,0.0,Meta #speculative,Meta #speculative,- [QTS Eagle Mountain website](https://q.com/data-center...,https://docs.google.com/spreadsheets/d/1pWiTKivojS1IY4ti...,NaN,NaN,B300,NaN,NaN,NaN,United States,"652 E Hyperscale Way, Eagle Mountain, UT 84013"


In [5]:
# Display the shape of the site table.
data_centers_df.shape


(96, 16)

In [6]:
# Display the first rows of the timeline table.
timelines_df.head()


,Data center,Date,Construction status,Buildings operational,IT power (MW),Power (MW),H100 equivalents,Performance (8-bit OP/s),Total capital cost (2025 USD billions),Compute cost (2025 USD billions),Construction cost (2025 USD billions),Annual operating cost (2025 USD billions),Water use (MGD),Current H100 equivalents (from Data center)
0,Colossus 2,2025-02-28,Land clearing begins next to the existing building.,0.0,0.0,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,NaN,1.111673e+06
1,Colossus 2,2025-09-08,Battery facility connected to the building. 121 air-cool...,0.0,0.0,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,NaN,1.111673e+06
2,Colossus 2,2025-10-19,Battery facility looks complete (though more might be ad...,1.0,239.0,323.0,277918.140475,5.500000e+20,9.053798,6.241007,2.812791,0.215578,NaN,1.111673e+06
3,Colossus 2,2026-01-13,"Since October, an array of dry cooling air-cooled conden...",1.0,239.0,323.0,277918.140475,5.500000e+20,9.053798,6.241007,2.812791,0.215578,NaN,1.111673e+06
4,Colossus 2,2026-04-06,The first array of dry cooling air-cooled condensers for...,1.0,490.0,661.0,555836.280950,1.100000e+21,18.562180,12.795370,5.766810,0.441980,NaN,1.111673e+06


In [7]:
# Display the last rows of the timeline table.
timelines_df.tail()


,Data center,Date,Construction status,Buildings operational,IT power (MW),Power (MW),H100 equivalents,Performance (8-bit OP/s),Total capital cost (2025 USD billions),Compute cost (2025 USD billions),Construction cost (2025 USD billions),Annual operating cost (2025 USD billions),Water use (MGD),Current H100 equivalents (from Data center)
570,QTS Eagle Mountain,2023-10-01,First signs of site preparation show up on satellite ima...,0.0,0.0,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,NaN,0.0
571,QTS Eagle Mountain,2025-09-08,Site preparation continues. Outline of a first building ...,0.0,0.0,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,NaN,0.0
572,QTS Eagle Mountain,2026-05-02,Building 1 roof finished. Satellite imagery shows constr...,0.0,0.0,0.0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,NaN,0.0
573,QTS Eagle Mountain,2026-12-31,We estimate Buildings 1 and 2 will be fully operational ...,2.0,203.0,274.0,231311.268317,4.577650e+20,7.690046,5.300939,2.389107,0.183106,NaN,0.0
574,QTS Eagle Mountain,2027-07-01,We estimate Building 3 will be fully operational by H2 2...,3.0,304.0,411.0,346968.165740,6.866500e+20,11.516128,7.938352,3.577776,0.274208,NaN,0.0


In [8]:
# Display the shape of the timeline table.
timelines_df.shape


(575, 14)

In [9]:
# Use .info() to inspect the data storage types and non-missing values.
# Count missing values in every feature.
data_centers_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 16 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Name                                            96 non-null     object 
 1   Current H100 equivalents                        96 non-null     float64
 2   Current power (MW)                              96 non-null     float64
 3   Current total capital cost (2025 USD billions)  96 non-null     float64
 4   Owner                                           86 non-null     object 
 5   Users                                           71 non-null     object 
 6   Selected Sources                                96 non-null     object 
 7   Calculations sheet                              96 non-null     object 
 8   Project                                         17 non-null     object 
 9   Current chip types                           

In [10]:
# Count missing values in every feature of the site table.
data_centers_df.isnull().sum()


Name                                               0
Current H100 equivalents                           0
Current power (MW)                                 0
Current total capital cost (2025 USD billions)     0
Owner                                             10
Users                                             25
Selected Sources                                   0
Calculations sheet                                 0
Project                                           79
Current chip types                                24
All chip types                                    10
Investors                                         86
Construction companies                            90
Energy companies                                  77
Country                                            0
Address                                           14
dtype: int64

In [11]:
# Inspect the storage types and non-missing values of the timeline table.
timelines_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 575 entries, 0 to 574
Data columns (total 14 columns):
 #   Column                                       Non-Null Count  Dtype         
---  ------                                       --------------  -----         
 0   Data center                                  575 non-null    object        
 1   Date                                         575 non-null    datetime64[ns]
 2   Construction status                          575 non-null    object        
 3   Buildings operational                        574 non-null    float64       
 4   IT power (MW)                                575 non-null    float64       
 5   Power (MW)                                   575 non-null    float64       
 6   H100 equivalents                             572 non-null    float64       
 7   Performance (8-bit OP/s)                     572 non-null    float64       
 8   Total capital cost (2025 USD billions)       575 non-null    float64       
 9  

In [12]:
# Count missing values in every feature of the timeline table.
timelines_df.isnull().sum()


Data center                                      0
Date                                             0
Construction status                              0
Buildings operational                            1
IT power (MW)                                    0
Power (MW)                                       0
H100 equivalents                                 3
Performance (8-bit OP/s)                         3
Total capital cost (2025 USD billions)           0
Compute cost (2025 USD billions)                 0
Construction cost (2025 USD billions)            0
Annual operating cost (2025 USD billions)        0
Water use (MGD)                                491
Current H100 equivalents (from Data center)      0
dtype: int64

In [13]:
# Check the coverage: countries in the site table and the date range of the timeline table.
display(data_centers_df["Country"].value_counts())
print("Countries other than the United States:", data_centers_df.loc[data_centers_df["Country"] != "United States", "Country"].nunique())
print("Rows and columns in the state-code lookup:", state_codes_df.shape, "missing values:", state_codes_df.isnull().sum().sum())

print("Earliest timeline date:", timelines_df["Date"].min().date())
print("Latest timeline date:", timelines_df["Date"].max().date())
print("Sites in the site table:", data_centers_df["Name"].nunique())
print("Sites that appear in the timeline table:", timelines_df["Data center"].nunique())
print("Timeline sites with no match in the site table:", (~timelines_df["Data center"].isin(data_centers_df["Name"])).sum())


Country
United States           77
China                    5
Malaysia                 4
Norway                   2
United Kingdom           1
Indonesia                1
Finland                  1
Australia                1
Portugal                 1
Iceland                  1
Sweden                   1
United Arab Emirates     1
Name: count, dtype: int64

Countries other than the United States: 11
Rows and columns in the state-code lookup: (50, 2) missing values: 0
Earliest timeline date: 2018-12-18
Latest timeline date: 2030-01-01
Sites in the site table: 96
Sites that appear in the timeline table: 96
Timeline sites with no match in the site table: 0


**Complete the table for every feature used in the question, preparation, or figures. Add rows as needed.**

| Feature | Meaning and units or codes | Feature type | Data storage type | Missing values |
| --- | --- | --- | --- | --- |
| `Name` | Site name in the site table. Matches `Data center` in the timeline table | categorical | object | 0 |
| `Country` | Country where the site is located | categorical | object | 0 |
| `Address` | Street address from public records, as free text | categorical | object | 14 |
| `Current power (MW)` | The site's latest IT power in megawatts | quantitative | float64 | 0 |
| `Data center` | Site name in the timeline table | categorical | object | 0 |
| `Date` | As-of date for the status and measures in a timeline record | temporal | datetime64[ns] | 0 |
| `IT power (MW)` | Total IT power capacity in megawatts (servers, networking, storage), not counting cooling and other overhead | quantitative | float64 | 0 |
| `state_name` | State name, from the lookup file | categorical | object | 0 |
| `state_code` | Two-letter state code, from the lookup file | categorical | object | 0 |
| `state` (created) | Two-letter state code found from `Address` | categorical | object | 0 |
| `quarter_end` (created) | Last day of the quarter that a value is measured on | temporal | datetime64[ns] | 0 |
| `quarter` (created) | Quarter label, such as 2026Q3 | temporal | object | 0 |
| `it_power_mw` (created) | Sum of IT power in megawatts across a state's tracked sites at a quarter end | quantitative | float64 | 0 |
| `sites_tracked` (created) | Number of tracked sites in the state | quantitative | int64 | 0 |
| `sites_with_power` (created) | Number of the state's tracked sites above 0 MW at that quarter end | quantitative | int64 | 0 |

Use quantitative, categorical, or temporal for feature type.

Your answer:

The first four features come from the site table, the next three from the timeline table, and the next two from the lookup file. The six created features are in `state_quarter_df`, the table behind both figures, and their counts are for that table. `Address` is text that I use only to find the state. `Current power (MW)` is used only to measure how much capacity is left off the state figures.


**Which observations and features will you use? Explain your time periods, geographic grouping, filtering, missing-value decisions, and any counts, means, sums, or rates.**

Your answer:

**Places.** I use the 77 United States sites and group them by state. The site table has no state feature, so I find the state from the address with three rules, each checked against the lookup file: a two-letter code before a ZIP code, a two-letter code at the end of the address, or a full state name in the address. I did not guess a state from a site's name.

**Time.** I keep timeline records dated on or before the access date, November 12, 2026, because later records are projections. I then take 15 quarter ends, from March 31, 2023 to September 30, 2026. For each site and quarter end, the value is the IT power in the site's most recent record on or before that date.

**Summary.** For each state and quarter end I sum the IT power of that state's sites. This is a sum of capacity, not a count of sites and not a rate.

**Missing values.** Ten United States sites have no usable state (8 with no address and 2 whose address has no state in it). They are left off the state figures and are not counted as zero for any state. A site with no record yet on a given quarter end is counted as 0 MW for that quarter, which I check in Step 4.


In [14]:
# Prepare the analysis data and summaries. Sort time values within each relevant group.
# Report the original and retained row counts and any unmatched geographic identifiers.
# Keep the United States sites. The site table has no state feature, so the state must come from the address.
us_sites_df = data_centers_df.loc[
    data_centers_df["Country"] == "United States",
    ["Name", "Address", "Current power (MW)"]
].copy()

print("Sites in the dataset:", len(data_centers_df))
print("United States sites kept:", len(us_sites_df))
print("Sites in other countries removed:", len(data_centers_df) - len(us_sites_df))


Sites in the dataset: 96
United States sites kept: 77
Sites in other countries removed: 19


In [15]:
# Find each site's state from its address with three rules, checked against the state-code lookup.
valid_codes = set(state_codes_df["state_code"])

# Rule 1: a two-letter code directly before a five-digit ZIP code, as in "Memphis, TN 38109".
from_zip = us_sites_df["Address"].str.extract(r"\b([A-Z]{2}),?\s+\d{5}")[0]
us_sites_df["state_from_zip"] = from_zip.where(from_zip.isin(valid_codes))

# Rule 2: a two-letter code at the very end of the address, as in "The Dalles, OR".
from_end = us_sites_df["Address"].str.strip().str.extract(r",\s*([A-Z]{2})$")[0]
us_sites_df["state_from_end"] = from_end.where(from_end.isin(valid_codes))

# Rule 3: a full state name written in the address, as in "Ridgeland, Mississippi".
# Longer names are tried first so that "West Virginia" is not read as "Virginia".
us_sites_df["state_from_name"] = pd.NA
longest_first = state_codes_df.sort_values("state_name", key=lambda names: names.str.len(), ascending=False)
for state_name, state_code in zip(longest_first["state_name"], longest_first["state_code"]):
    has_name = us_sites_df["Address"].str.contains(rf"\b{state_name}\b", na=False)
    us_sites_df.loc[has_name & us_sites_df["state_from_name"].isna(), "state_from_name"] = state_code

us_sites_df["state"] = (
    us_sites_df["state_from_zip"]
    .fillna(us_sites_df["state_from_end"])
    .fillna(us_sites_df["state_from_name"])
)

us_sites_df["state_source"] = "address has no state"
us_sites_df.loc[us_sites_df["Address"].isna(), "state_source"] = "no address recorded"
us_sites_df.loc[us_sites_df["state_from_name"].notna(), "state_source"] = "state name in address"
us_sites_df.loc[us_sites_df["state_from_end"].notna(), "state_source"] = "state code at end of address"
us_sites_df.loc[us_sites_df["state_from_zip"].notna(), "state_source"] = "state code before ZIP code"

us_sites_df["state_source"].value_counts()


state_source
state code before ZIP code      59
no address recorded              8
state name in address            4
state code at end of address     4
address has no state             2
Name: count, dtype: int64

In [16]:
# Display the sites matched by rules 2 and 3 so the matches can be read against the addresses.
us_sites_df.loc[
    us_sites_df["state_source"].isin(["state code at end of address", "state name in address"]),
    ["Name", "Address", "state", "state_source"]
]


,Name,Address,state,state_source
19,Amazon Madison Mega Site,"Amazon Data Services Inc, Madison Mega Site Nissan Parkw...",MS,state name in address
29,Google The Dalles,"3500 River Rd, The Dalles, OR",OR,state code at end of address
34,Amazon Ridgeland,"1626 County Line Road Ridgeland, Mississippi",MS,state name in address
35,Meta Huntsville,"5400 Prosperity Dr NW, Toney, AL",AL,state code at end of address
50,Google Papillion,"14651-15249 Gold Coast Rd 68138 Papillion Nebraska, USA",NE,state name in address
54,CoreWeave Chester VA,"1401 Meadowville Technology Parkway, Chester, VA",VA,state code at end of address
62,Microsoft-Nebius New Jersey,"3963 S Lincoln Ave, Vineland, New Jersey",NJ,state name in address
77,CoreWeave Lancaster Greenfield site,"216 Greenfield Rd, Lancaster, PA \n",PA,state code at end of address


In [17]:
# Report the United States sites with no usable state and how much current power they hold.
unmatched_sites = us_sites_df.loc[us_sites_df["state"].isna(), ["Name", "Address", "state_source", "Current power (MW)"]]
display(unmatched_sites)

matched_power = us_sites_df.loc[us_sites_df["state"].notna(), "Current power (MW)"].sum()
unmatched_power = unmatched_sites["Current power (MW)"].sum()
print("United States sites with a state:", us_sites_df["state"].notna().sum())
print("United States sites without a state:", len(unmatched_sites))
print(f"Current power at sites with a state: {matched_power:,.0f} MW")
print(f"Current power at sites without a state: {unmatched_power:,.0f} MW")
print(f"Share of United States current power left off the state figures: {unmatched_power / (matched_power + unmatched_power) * 100:.1f}%")


,Name,Address,state_source,Current power (MW)
12,Google Mesa,NaN,no address recorded,183.0
23,Google Kansas City East,NaN,no address recorded,105.0
25,AWS New Albany,13360 Miller Rd NW,address has no state,213.0
42,Google Storey County,NaN,no address recorded,161.0
68,Stream Phoenix,2950 S. Litchfield Road,address has no state,64.0
76,Anthropic Barber Lake,NaN,no address recorded,0.0
88,OpenAI Stargate Michigan,NaN,no address recorded,0.0
89,OpenAI Stargate Milam,NaN,no address recorded,0.0
90,OpenAI Stargate New Mexico,NaN,no address recorded,0.0
93,OpenAI Stargate Wisconsin,NaN,no address recorded,0.0


United States sites with a state: 67
United States sites without a state: 10
Current power at sites with a state: 11,914 MW
Current power at sites without a state: 726 MW
Share of United States current power left off the state figures: 5.7%


In [18]:
# Keep timeline records dated on or before the access date. Later dates are projections.
access_date = pd.Timestamp("2026-11-12")
history_df = timelines_df.loc[timelines_df["Date"] <= access_date].copy()

print("Timeline records in the dataset:", len(timelines_df))
print("Records dated on or before November 12, 2026:", len(history_df))
print("Projected records removed:", len(timelines_df) - len(history_df))


Timeline records in the dataset: 575
Records dated on or before November 12, 2026: 481
Projected records removed: 94


In [19]:
# Build one row per site and quarter end: the IT power in the site's most recent record on or before that date.
quarter_ends = pd.date_range("2023-03-31", "2026-09-30", freq="QE")
sites_with_state = us_sites_df.loc[us_sites_df["state"].notna(), ["Name", "state"]]

site_quarter_frames = []
for quarter_end in quarter_ends:
    latest_records = (
        history_df.loc[history_df["Date"] <= quarter_end]
        .sort_values("Date")
        .groupby("Data center")
        .tail(1)[["Data center", "IT power (MW)"]]
    )
    site_quarter = sites_with_state.merge(latest_records, left_on="Name", right_on="Data center", how="left")
    # A site with no record yet on this date is counted as 0 MW.
    site_quarter["IT power (MW)"] = site_quarter["IT power (MW)"].fillna(0)
    site_quarter["quarter_end"] = quarter_end
    site_quarter_frames.append(site_quarter)

site_quarter_df = pd.concat(site_quarter_frames, ignore_index=True)
print("Quarter ends:", len(quarter_ends), "from", quarter_ends[0].date(), "to", quarter_ends[-1].date())
print("Site-quarter rows:", len(site_quarter_df))


Quarter ends:

 15 from 2023-03-31 to 2026-09-30
Site-quarter rows: 1005


In [20]:
# Sum to one row per state and quarter end, sorted by state and then by time.
state_quarter_df = (
    site_quarter_df.groupby(["state", "quarter_end"])
    .agg(
        it_power_mw=("IT power (MW)", "sum"),
        sites_tracked=("Name", "count"),
        sites_with_power=("IT power (MW)", lambda power: (power > 0).sum())
    )
    .reset_index()
    .merge(state_codes_df, left_on="state", right_on="state_code", how="left")
    .sort_values(["state", "quarter_end"])
    .reset_index(drop=True)
)
state_quarter_df["quarter"] = state_quarter_df["quarter_end"].dt.to_period("Q").astype(str)

print("States with at least one tracked site:", state_quarter_df["state"].nunique())
print("State-quarter rows:", len(state_quarter_df))
print("State codes with no match in the lookup:", state_quarter_df["state_name"].isna().sum())
state_quarter_df.tail()


States with at least one tracked site:

 26
State-quarter rows: 390
State codes with no match in the lookup: 0


,state,quarter_end,it_power_mw,sites_tracked,sites_with_power,state_name,state_code,quarter
385,WY,2025-09-30,0.0,1,0,Wyoming,WY,2025Q3
386,WY,2025-12-31,0.0,1,0,Wyoming,WY,2025Q4
387,WY,2026-03-31,0.0,1,0,Wyoming,WY,2026Q1
388,WY,2026-06-30,0.0,1,0,Wyoming,WY,2026Q2
389,WY,2026-09-30,152.0,1,1,Wyoming,WY,2026Q3


In [21]:
# Inspect the storage types and non-missing values of the state-by-quarter table behind both figures.
state_quarter_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   state             390 non-null    object        
 1   quarter_end       390 non-null    datetime64[ns]
 2   it_power_mw       390 non-null    float64       
 3   sites_tracked     390 non-null    int64         
 4   sites_with_power  390 non-null    int64         
 5   state_name        390 non-null    object        
 6   state_code        390 non-null    object        
 7   quarter           390 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 24.5+ KB


**How many records were removed from the dataset? Explain why, and state what one row represents after any aggregation.**

Your answer:

- 19 of the 96 sites were removed because they are outside the United States, which is outside the question.
- 10 of the 77 United States sites were left off the state figures because no state could be found in the address. They hold 726 MW of current power, 5.7% of the United States total.
- 94 of the 575 timeline records were removed because they are dated after November 12, 2026 and are projections.

After aggregation, one row is one state at one quarter end, holding the summed IT power of that state's tracked sites. There are 26 states and 15 quarter ends, so 390 rows.


### Step 3: Evidence

Create two different Plotly Express figure types that answer this same question. Together they must display the spatial and temporal patterns. Use the prepared data consistently and explain any difference in coverage between the figures.

**Which two chart types will you use, and what part of the question should each make easier to answer? Specify the time, location, and quantitative encodings.**

Your answer:

**Figure 1 is a choropleth map** (`px.choropleth`). It answers where. Location is the state, drawn from its two-letter code. Color is IT power in MW. Time is held at one date, September 30, 2026, the latest quarter end.

**Figure 2 is a location-by-time heatmap** (`px.imshow`). It answers when. Each row is a state, ordered from the largest to the smallest in the latest quarter. Each column is a quarter, in time order from 2023 Q1 to 2026 Q3. Color is IT power in MW at the end of that quarter.

Both figures use the same state-by-quarter table and the same color scale, so the map is the last column of the heatmap placed on geography.


In [22]:
# Create Figure 1 with a descriptive title, units, useful hover fields, and suitable dimensions.
# If animated, use chronological frames and fixed scales.
latest_quarter_df = state_quarter_df.loc[state_quarter_df["quarter_end"] == quarter_ends[-1]]

# Use one color range, from 0 to the largest state-quarter value, in both figures.
power_max = state_quarter_df["it_power_mw"].max()

fig = px.choropleth(
    latest_quarter_df,
    locations="state",
    locationmode="USA-states",
    scope="usa",
    color="it_power_mw",
    color_continuous_scale="Viridis",
    range_color=[0, power_max],
    hover_name="state_name",
    hover_data={"state": False, "it_power_mw": ":,.0f", "sites_tracked": True, "sites_with_power": True, "quarter": True},
    labels={
        "it_power_mw": "IT power (MW)",
        "sites_tracked": "Tracked sites",
        "sites_with_power": "Sites with power",
        "quarter": "Quarter"
    },
    title="Estimated IT Power of Tracked AI Data Centers by State, September 30, 2026"
)
fig.update_layout(height=500)
fig.show()


In [23]:
# List the states behind the map, largest first, with each state's share of the mapped total.
latest_ranked = latest_quarter_df.sort_values("it_power_mw", ascending=False)[
    ["state_name", "it_power_mw", "sites_tracked", "sites_with_power"]
].reset_index(drop=True)
latest_ranked["share_of_total_percent"] = (latest_ranked["it_power_mw"] / latest_ranked["it_power_mw"].sum() * 100).round(1)
latest_ranked["it_power_mw"] = latest_ranked["it_power_mw"].round(0)
display(latest_ranked)

print(f"Mapped total on September 30, 2026: {latest_quarter_df['it_power_mw'].sum():,.0f} MW")
print(f"Share held by the five largest states: {latest_ranked['share_of_total_percent'].head(5).sum():.1f}%")
print("States with power above 0 MW:", (latest_quarter_df["it_power_mw"] > 0).sum(), "of", len(latest_quarter_df))


,state_name,it_power_mw,sites_tracked,sites_with_power,share_of_total_percent
0,Texas,1524.0,12,9,13.5
1,Tennessee,1373.0,3,3,12.1
2,Ohio,1244.0,6,4,11.0
3,Indiana,1088.0,3,2,9.6
4,Virginia,1044.0,7,7,9.2
5,Nebraska,695.0,4,4,6.1
6,Georgia,681.0,3,3,6.0
7,Mississippi,512.0,2,2,4.5
8,Iowa,427.0,4,2,3.8
9,Wisconsin,369.0,1,1,3.3


Mapped total on September 30, 2026: 11,322 MW
Share held by the five largest states: 55.4%
States with power above 0 MW: 25 of 26


**What does Figure 1 show? Describe a pattern and record at least one value with its place, time, and units or count.**

Your answer:

The map shows that the tracked capacity is concentrated in a few states. Texas has the most, with 1,524 MW on September 30, 2026, followed by Tennessee (1,373 MW), Ohio (1,244 MW), Indiana (1,088 MW), and Virginia (1,044 MW). Those five states hold 55.4% of the 11,322 MW on the map.

The brightest states run from Texas up through Tennessee, Indiana, and Ohio to Virginia. Twenty-five of the 26 states with a tracked site show power above 0 MW. Louisiana has one tracked site, which is still at 0 MW. States with no color have no tracked site matched to them. That is not the same as having no data centers.


In [24]:
# Create Figure 2 using a different chart type to answer the same research question.
# Label any aggregation or transformation.
heatmap_grid = state_quarter_df.pivot(index="state_name", columns="quarter", values="it_power_mw")
heatmap_grid = heatmap_grid.reindex(sorted(heatmap_grid.columns), axis=1)

# Order the rows from the largest to the smallest state in the latest quarter.
heatmap_grid = heatmap_grid.loc[heatmap_grid["2026Q3"].sort_values(ascending=False).index]

fig = px.imshow(
    heatmap_grid,
    aspect="auto",
    color_continuous_scale="Viridis",
    zmin=0,
    zmax=power_max,
    labels={"x": "Quarter (value at quarter end)", "y": "State", "color": "IT power (MW)"},
    title="Estimated IT Power of Tracked AI Data Centers by State and Quarter, 2023 Q1 to 2026 Q3"
)
fig.update_layout(height=700)
fig.show()


In [25]:
# Total the tracked states at each year end and at the latest quarter.
national_by_quarter = state_quarter_df.groupby("quarter")["it_power_mw"].sum()
display(national_by_quarter.loc[["2023Q1", "2023Q4", "2024Q4", "2025Q4", "2026Q3"]].round(0))

print(f"Added from 2023 Q1 to 2026 Q3: {national_by_quarter['2026Q3'] - national_by_quarter['2023Q1']:,.0f} MW")
print(f"Added from 2025 Q4 to 2026 Q3: {national_by_quarter['2026Q3'] - national_by_quarter['2025Q4']:,.0f} MW")
print(f"Share of the 2026 Q3 total added after 2025 Q4: {(national_by_quarter['2026Q3'] - national_by_quarter['2025Q4']) / national_by_quarter['2026Q3'] * 100:.1f}%")


quarter
2023Q1      127.0
2023Q4      259.0
2024Q4     1617.0
2025Q4     5372.0
2026Q3    11322.0
Name: it_power_mw, dtype: float64

Added from 2023 Q1 to 2026 Q3: 11,195 MW
Added from 2025 Q4 to 2026 Q3: 5,950 MW
Share of the 2026 Q3 total added after 2025 Q4: 52.6%


In [26]:
# Find the first quarter in which each state shows power above 0 MW, and count states with power at each year end.
first_power_quarter = (
    state_quarter_df.loc[state_quarter_df["it_power_mw"] > 0]
    .groupby("state_name")["quarter"]
    .min()
    .sort_values()
)
display(first_power_quarter)

states_with_power = (
    state_quarter_df.loc[state_quarter_df["it_power_mw"] > 0]
    .groupby("quarter")["state"]
    .nunique()
)
display(states_with_power.loc[["2023Q1", "2023Q4", "2024Q4", "2025Q4", "2026Q3"]])


state_name
Nebraska          2023Q1
Iowa              2023Q1
Arizona           2023Q2
Oregon            2023Q3
Ohio              2024Q1
Virginia          2024Q1
Texas             2024Q2
Oklahoma          2024Q3
Tennessee         2024Q3
Georgia           2025Q1
Indiana           2025Q2
Mississippi       2025Q2
Pennsylvania      2025Q3
Utah              2025Q4
New York          2025Q4
North Dakota      2025Q4
Alabama           2026Q1
North Carolina    2026Q1
New Mexico        2026Q1
New Jersey        2026Q2
Minnesota         2026Q2
Idaho             2026Q2
Wisconsin         2026Q2
South Carolina    2026Q3
Wyoming           2026Q3
Name: quarter, dtype: object

quarter
2023Q1     2
2023Q4     4
2024Q4     9
2025Q4    16
2026Q3    25
Name: state, dtype: int64

In [27]:
# Read selected values from the heatmap grid for the largest, earliest, and smallest states.
heatmap_grid.loc[
    ["Texas", "Tennessee", "Ohio", "Indiana", "Virginia", "Nebraska", "Iowa", "New Jersey", "Louisiana"],
    ["2023Q1", "2023Q4", "2024Q4", "2025Q2", "2025Q3", "2025Q4", "2026Q2", "2026Q3"]
].round(0)


quarter,2023Q1,2023Q4,2024Q4,2025Q2,2025Q3,2025Q4,2026Q2,2026Q3
state_name,,,,,,,,
Texas,0.0,0.0,66.0,74.0,254.0,596.0,995.0,1524.0
Tennessee,0.0,0.0,139.0,278.0,268.0,579.0,1286.0,1373.0
Ohio,0.0,0.0,277.0,420.0,420.0,762.0,1141.0,1244.0
Indiana,0.0,0.0,0.0,398.0,398.0,626.0,1088.0,1088.0
Virginia,0.0,0.0,293.0,548.0,548.0,668.0,1044.0,1044.0
Nebraska,79.0,79.0,395.0,395.0,395.0,395.0,615.0,695.0
Iowa,48.0,95.0,174.0,348.0,348.0,348.0,348.0,427.0
New Jersey,0.0,0.0,0.0,0.0,0.0,0.0,50.0,50.0
Louisiana,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**What does Figure 2 add? Record at least one value with its place, time, and units or count, and compare its message with Figure 1.**

Your answer:

The heatmap adds the timing, and it shows that almost all of this capacity is recent. The tracked total was 127 MW at the end of 2023 Q1 and 259 MW at the end of 2023. It reached 1,617 MW at the end of 2024, 5,372 MW at the end of 2025, and 11,322 MW at the end of 2026 Q3. More than half of the 2026 Q3 total, 52.6%, was added after the end of 2025.

The number of states with power grew the same way: 2 states in 2023 Q1, 9 at the end of 2024, 16 at the end of 2025, and 25 in 2026 Q3.

The heatmap also changes how the map should be read. Texas is the brightest state on the map, but it had 0 MW at the end of 2023 and only 66 MW at the end of 2024. Nebraska and Iowa are the two states with power in 2023 Q1, and at the end of 2024 Nebraska (395 MW) was still far ahead of Texas. The map shows who leads now, and the heatmap shows that the order was different two years earlier.


### Step 4: Check

Use hover to compare at least two plotted values with the source records or a calculated summary. Check chronological order, location matching, and one unusual high, low, or apparent gap that matters to your question. Explain any problem you corrected. Do not repeat the initial missing-value count.

In [28]:
# Verify selected plotted values against records or summaries.
# Check the relevant time order, geographic match, or unusual observation.
# Check 1: rebuild the Texas value for 2026 Q3 directly from the timeline records.
texas_sites = us_sites_df.loc[us_sites_df["state"] == "TX", "Name"]
texas_latest = (
    timelines_df.loc[
        timelines_df["Data center"].isin(texas_sites) & (timelines_df["Date"] <= pd.Timestamp("2026-09-30")),
        ["Data center", "Date", "IT power (MW)"]
    ]
    .sort_values("Date")
    .groupby("Data center")
    .tail(1)
    .sort_values("IT power (MW)", ascending=False)
)
display(texas_latest)

print("Texas sites with a state:", len(texas_sites))
print("Texas sites with a record on or before September 30, 2026:", len(texas_latest))
print(f"Sum of those records: {texas_latest['IT power (MW)'].sum():,.0f} MW")
print(f"Value plotted for Texas in 2026 Q3: {heatmap_grid.loc['Texas', '2026Q3']:,.0f} MW")


,Data center,Date,IT power (MW)
15,OpenAI Stargate Abilene,2026-09-10,618.0
124,CoreWeave Denton TX,2026-09-01,262.0
181,Meta Temple,2026-09-30,178.0
249,Coreweave Helios,2026-08-30,132.0
275,Google Midlothian,2026-07-01,103.0
317,Google Red Oak,2026-02-06,77.0
386,Microsoft SAT40,2026-02-04,75.0
433,Microsoft SAT14,2026-03-31,47.0
435,Vantage TX1,2026-02-15,32.0
475,Crusoe Abilene Expansion,2026-01-15,0.0


Texas sites with a state: 12
Texas sites with a record on or before September 30, 2026: 12
Sum of those records: 1,524 MW
Value plotted for Texas in 2026 Q3: 1,524 MW


In [29]:
# Check 2: Tennessee is the one state whose value falls between two quarters. Read its site records.
tennessee_sites = us_sites_df.loc[us_sites_df["state"] == "TN", "Name"]
tennessee_records = timelines_df.loc[
    timelines_df["Data center"].isin(tennessee_sites)
    & (timelines_df["Date"] >= "2025-01-01") & (timelines_df["Date"] <= "2025-12-31"),
    ["Data center", "Date", "IT power (MW)"]
].sort_values(["Data center", "Date"])
display(tennessee_records)

display(heatmap_grid.loc[["Tennessee"], ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]].round(0))

quarter_changes = heatmap_grid.diff(axis=1)
print("State-quarter steps where the value falls:", int((quarter_changes < 0).sum().sum()))


,Data center,Date,IT power (MW)
102,Colossus 1,2025-02-17,278.0
103,Colossus 1,2025-03-06,278.0
104,Colossus 1,2025-07-22,340.0
105,Colossus 1,2025-09-17,268.0
106,Colossus 1,2025-10-19,268.0
107,Colossus 1,2025-11-07,340.0
0,Colossus 2,2025-02-28,0.0
1,Colossus 2,2025-09-08,0.0
2,Colossus 2,2025-10-19,239.0


quarter,2025Q1,2025Q2,2025Q3,2025Q4
state_name,,,,
Tennessee,278.0,278.0,268.0,579.0


State-quarter steps where the value falls: 1


In [30]:
# Check 3: time order, location matching, and the 0 MW assumption for quarters before a site's first record.
print("Heatmap quarters in order:", list(heatmap_grid.columns))
print("Quarter ends increase from left to right:", quarter_ends.is_monotonic_increasing)
print("Every mapped state code is in the lookup:", state_quarter_df["state"].isin(valid_codes).all())

first_records = (
    timelines_df.loc[timelines_df["Data center"].isin(sites_with_state["Name"])]
    .sort_values("Date")
    .groupby("Data center")
    .head(1)[["Data center", "Date", "IT power (MW)"]]
)
print("Mapped sites whose first record is 0 MW:", (first_records["IT power (MW)"] == 0).sum(), "of", len(first_records))
display(first_records.loc[first_records["IT power (MW)"] > 0])


Heatmap quarters in order: ['2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2', '2026Q3']
Quarter ends increase from left to right: True
Every mapped state code is in the lookup: True
Mapped sites whose first record is 0 MW: 66 of 67


,Data center,Date,IT power (MW)
34,Google Pryor (North),2024-07-19,53.0


**Which two values did you check, and did they agree? What did your check establish about order, location, or an unusual pattern?**

Your answer:

**Value 1: Texas, 2026 Q3.** The figures show 1,524 MW. I rebuilt it from the timeline table by taking each of the 12 Texas sites' most recent record on or before September 30, 2026. The 12 records sum to 1,524 MW, so the plotted value agrees.

**Value 2: Tennessee, 2025 Q3.** The heatmap shows Tennessee falling from 278 MW in 2025 Q2 to 268 MW in 2025 Q3, which is the only step in the whole grid where a state's value goes down. The source records explain it. The Colossus 1 site is recorded at 278 MW on March 6, 2025, 340 MW on July 22, 268 MW on September 17, and 340 MW again on November 7. The plotted values agree with the records, and the drop is in the source data, not an error in my preparation.

**Order and location.** The 15 quarters are in time order from left to right, and every mapped state code is in the lookup file.

**The 0 MW assumption.** For 66 of the 67 mapped sites, the first timeline record is at 0 MW, so counting earlier quarters as 0 MW is reasonable. The exception is Google Pryor (North) in Oklahoma, whose first record is 53 MW on July 19, 2024. Oklahoma's values before 2024 Q3 may therefore be too low. I left the data as it is and note it as a limitation.


### Step 5: Conclusion

Restate your research question, then answer it using evidence from **both** figures. Include a comparison across places and a comparison across times, with values and units. Explain whether the evidence supports your prediction.

**Restate your question and write your conclusion.**

Your answer:

**Question.** Where in the United States has the IT power capacity of large AI data centers been built, and when did each state's capacity come online, between 2023 Q1 and 2026 Q3?

**Where.** The capacity is concentrated. On September 30, 2026 the map shows Texas with 1,524 MW, Tennessee with 1,373 MW, Ohio with 1,244 MW, Indiana with 1,088 MW, and Virginia with 1,044 MW, and those five states hold 55.4% of the 11,322 MW that could be placed in a state. At the other end, New Jersey has 50 MW and Louisiana 0 MW.

**When.** Most of it is recent. The heatmap shows the tracked total rising from 259 MW at the end of 2023 to 1,617 MW at the end of 2024, 5,372 MW at the end of 2025, and 11,322 MW at the end of 2026 Q3, with 52.6% of that final total added after 2025. The states also came online at different times. Nebraska and Iowa had power in 2023 Q1, Texas had none until 2024 Q2, and Indiana had none until 2025 Q2 and then reached 1,088 MW by 2026 Q2.

**Prediction.** The evidence supports part of it. Texas is first and most of the capacity did arrive in 2025 and 2026. I was wrong to expect Virginia near the top. It is fifth, behind Tennessee, Ohio, and Indiana, which I did not expect.


### Step 6: Limitation

Identify **two limitations of your analysis**. Then explain which aspects of the research question are easier or harder to answer from each figure.

**What are two limitations, and how do the strengths and weaknesses of your figures affect the answer?**

Your answer:

**Limitation 1: these are estimates for a selected set of sites.** Epoch AI estimates IT power from satellite imagery, permits, and public documents, and it tracks large AI data centers, not all data centers. Ten United States sites holding 726 MW (5.7% of current power) could not be placed in a state and are missing from both figures. An uncolored state means no tracked site was matched to it.

**Limitation 2: quarter-end snapshots and capacity, not use.** Each value is the last record on or before a quarter end, so changes inside a quarter are not shown. Colossus 1's 340 MW in July 2025 never appears because it was recorded at 268 MW by the end of that quarter. Quarters before a site's first record are counted as 0 MW, which is too low for at least one Oklahoma site. The measure is installed IT power capacity. It is not electricity actually used, and it is not adjusted for a state's size or population.

**What each figure makes easier or harder.** The map makes the where easy: which states lead and how they sit next to each other. It shows one date, so it cannot answer when, and large states draw more attention than small ones with the same value. The heatmap makes the when easy: it shows each state's start and how fast it grew, and all 26 states can be compared at once. It loses the geography, because neighboring states are not next to each other in the rows, and small differences between dark cells in the early quarters are hard to read without hovering.


## Investigation 2: Your dataset and research question

### Step 1: Question

Ask one focused question that depends on **both where and when** something occurs. Name the measurement or count, the places, and the period. Record a prediction before producing the figures.

**What is your research question, and what do you predict? Why is it useful to consider both space and time?**

Your answer:

**Question.** Between November 18, 2025 and November 12, 2026, which countries had the most Internet outages recorded by Cloudflare Radar, and in which months did those outages happen? The count is the number of outage records, the places are countries, and the period is the last 12 months of Cloudflare's Outage Center as I downloaded it on November 15, 2026.

**Why this dataset.** I chose it because of the Microsoft 365 outage on November 12, 2026. I wanted to connect the lab to something happening in the world.

**Prediction.** I predict that the outages are concentrated in a small number of countries, and that the countries at the top are ones where the government shuts the Internet off or where there is a war. I do not expect them to be the countries with the most Internet users, such as the United States. I also predict that the outages are uneven across the year and come in clusters, because shutdowns and attacks tend to happen in runs of days.

**Why space and time together.** A total for each country cannot show whether its outages were one bad week or trouble all year. A total for each month cannot show where the outages were. A country with ten outages in ten days has a different problem than a country with ten outages spread over a year.


### Step 2: Data

Document the source and explain what one row represents. Load a fixed copy, inspect the first and last rows and shape, use `.info()` to inspect the data storage types and non-missing values, and count missing values. Parse dates when needed and preserve geographic identifiers appropriately.

**Give the dataset title, publisher, direct source URL, dictionary reference, and access date. Identify your saved file and any lookup or boundary files.**

Your answer:

- **Title:** Cloudflare Radar Outage Center, Internet outages table
- **Publisher:** Cloudflare, Inc. (Cloudflare Radar)
- **Direct source URL:** https://radar.cloudflare.com/charts/InternetOutagesTable/attachment?dateRange=52w, which is the Download CSV link on https://radar.cloudflare.com/outage-center?dateRange=52w
- **Dictionary reference:** https://developers.cloudflare.com/api/resources/radar/subresources/annotations/subresources/outages/methods/get/ for the field definitions, and https://developers.cloudflare.com/radar/glossary/ for how outages get listed. The CSV columns are the API fields, with the word Annotations in front and spaces added.
- **Access date:** November 15, 2026
- **License:** CC BY-NC 4.0, as stated on https://radar.cloudflare.com/about
- **Saved file:** `data/cloudflare-radar_outages_worldwide_20251116-20261115.csv`, exactly as downloaded. The only filter is the page's date range, set to the last 12 months.
- **Lookup file:** `data/iso_3166_country_codes.csv`, which is the `all.csv` file from https://github.com/lukes/ISO-3166-Countries-with-Regional-Codes, downloaded November 15, 2026 and not edited. Its documentation says it was built from the ISO 3166-1 code list and the United Nations M49 standard. It lists 249 countries and territories with their two-letter code, three-letter code, and name. I use it to turn Cloudflare's two-letter codes into the three-letter codes a Plotly map needs.
- **Boundary file:** none. Plotly Express draws the countries from the three-letter codes.


**What does one row represent? Which places and dates are covered, and how were the observations collected?**

Your answer:

One row is one outage that Cloudflare Radar recorded. An outage here is a notable drop in Internet traffic for a country, a region inside a country, a single network, or an online platform. The file has 140 rows and 27 features.

**Places.** 111 rows carry a two-letter country code, and those codes cover 49 countries and territories. The other 29 rows have no country code in that field. Of those, 21 are network or regional outages whose country is listed with the affected network or region, and 8 are platform outages, which Cloudflare records against a cloud service or its regions and not against a country.

**Dates.** The outages start between November 18, 2025 and November 12, 2026. The file name gives the download window as November 16, 2025 to November 15, 2026. All times are in UTC.

**Collection.** Cloudflare's glossary says the outages listed in the Outage Center are notable drops in traffic that have generally been corroborated with third-party information. This is a list that Cloudflare builds from what its own network sees and then confirms. It is not a complete record of every outage in the world.


In [31]:
# Read dataset 2 from a relative path into a descriptive DataFrame ending in _df.
# Parse the time feature as appropriate for its documented format.
# "NA" is the country code for Namibia, so only empty fields are read as missing.
outages_df = pd.read_csv(
    "data/cloudflare-radar_outages_worldwide_20251116-20261115.csv",
    keep_default_na=False,
    na_values=[""],
    parse_dates=["Annotations start Date", "Annotations end Date"]
)

# Shorten the names of the features this investigation uses.
outages_df = outages_df.rename(columns={
    "Annotations id": "outage_id",
    "Annotations start Date": "start_date",
    "Annotations end Date": "end_date",
    "Annotations locations": "location_code",
    "Annotations entities.entity Location": "entity_location_codes",
    "Annotations outage.outage Type": "outage_type",
    "Annotations outage.outage Cause": "outage_cause",
    "Annotations description": "description"
})

# Supporting lookup: ISO 3166-1 two-letter and three-letter country codes and country names.
country_codes_df = pd.read_csv("data/iso_3166_country_codes.csv", keep_default_na=False, na_values=[""])


In [32]:
# Display .head(), .tail(), and .shape in separate cells. Add cells as needed.
outages_df.head()


,outage_id,Annotations data Source,description,Annotations scope,start_date,end_date,location_code,Annotations asns,Annotations geo Ids,Annotations origins,...,Annotations origins Details.origin,outage_cause,outage_type,Annotations locations Details.name,Annotations locations Details.code,Annotations asns Details.asn,Annotations asns Details.name,Annotations asns Details.location,Annotations asns Details.location.code,Annotations asns Details.location.name
0,1749,HTTP_ORIGINS,A global Microsoft 365 disruption impacted Exchange Onli...,NaN,2026-11-12 16:17:00+00:00,NaT,NaN,NaN,NaN,microsoft-australiacentral;microsoft-australiacentral2;m...,...,microsoft-australiacentral;microsoft-australiacentral2;m...,TECHNICAL_PROBLEM,PLATFORM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1748,ALL,Russian strikes on energy infrastructure caused power ou...,NaN,2026-11-08 03:15:00+00:00,2026-11-08 21:00:00+00:00,UA,NaN,709929,NaN,...,NaN,POWER_OUTAGE,REGIONAL,Ukraine,UA,NaN,NaN,NaN,NaN,NaN
2,1747,ALL,A network fault caused a near-total loss of Internet tra...,NaN,2026-11-06 16:45:00+00:00,2026-11-06 18:45:00+00:00,DO,6400,NaN,NaN,...,NaN,TECHNICAL_PROBLEM,NETWORK,Dominican Republic,DO,6400,Claro Dominicana,NaN,DO,Dominican Republic
3,1746,ALL,A power plant failure at WEB Aruba caused a significant ...,NaN,2026-11-05 18:00:00+00:00,2026-11-05 20:15:00+00:00,AW,NaN,NaN,NaN,...,NaN,POWER_OUTAGE,NATIONWIDE,Aruba,AW,NaN,NaN,NaN,NaN,NaN
4,1744,ALL,Damage to energy facilities cut power in Zhytomyr and di...,NaN,2026-11-03 17:30:00+00:00,2026-11-04 07:00:00+00:00,NaN,NaN,686966,NaN,...,NaN,POWER_OUTAGE,REGIONAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
# Display the last rows of the outage table.
outages_df.tail()


,outage_id,Annotations data Source,description,Annotations scope,start_date,end_date,location_code,Annotations asns,Annotations geo Ids,Annotations origins,...,Annotations origins Details.origin,outage_cause,outage_type,Annotations locations Details.name,Annotations locations Details.code,Annotations asns Details.asn,Annotations asns Details.name,Annotations asns Details.location,Annotations asns Details.location.code,Annotations asns Details.location.name
135,1508,ALL,"Brief Internet disruption, possibly related to attempted...",NaN,2025-12-07 17:30:00+00:00,2025-12-07 18:30:00+00:00,BJ,28683;37136;37424,NaN,NaN,...,NaN,UNKNOWN,NATIONWIDE,Benin,BJ,28683;37136;37424,SBIN;ETISALAT-AS;Spacetel,[object Object];[object Object];[object Object],NaN,NaN
136,1507,ALL,Aceh and the Sumatra regions in Indonesia were impacted ...,NaN,2025-11-25 22:00:47+00:00,2025-12-11 21:00:00+00:00,ID,NaN,1213642;1215638,NaN,...,NaN,WEATHER,REGIONAL,Indonesia,ID,NaN,NaN,NaN,NaN,NaN
137,1505,ALL,Damage to international fiber optic infrastructure,NaN,2025-11-25 07:00:00+00:00,2025-11-25 13:00:00+00:00,HT,27653,NaN,NaN,...,NaN,CABLE_CUT,NETWORK,Haiti,HT,27653,Digicel Haiti,NaN,HT,Haiti
138,1506,ALL,"In Sri Lanka, because of last week’s Cyclone Senyar, reg...",NaN,2025-11-25 02:00:11+00:00,2025-12-09 00:00:00+00:00,LK,NaN,1225265;1227618;1228435;1232860;1232870;1249296;7671049,NaN,...,NaN,WEATHER,REGIONAL,Sri Lanka,LK,NaN,NaN,NaN,NaN,NaN
139,1468,ALL,Outage caused by heavy rains,NaN,2025-11-18 09:15:00+00:00,2025-11-18 15:00:00+00:00,BO,25620,NaN,NaN,...,NaN,WEATHER,NETWORK,Bolivia,BO,25620,COTAS LTDA.,NaN,BO,Bolivia


In [34]:
# Display the shape of the outage table.
outages_df.shape


(140, 27)

In [35]:
# Display the first rows and the shape of the country-code lookup.
display(country_codes_df.head())
country_codes_df.shape


,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
0,Afghanistan,AF,AFG,4,ISO 3166-2:AF,Asia,Southern Asia,NaN,142.0,34.0,NaN
1,Åland Islands,AX,ALA,248,ISO 3166-2:AX,Europe,Northern Europe,NaN,150.0,154.0,NaN
2,Albania,AL,ALB,8,ISO 3166-2:AL,Europe,Southern Europe,NaN,150.0,39.0,NaN
3,Algeria,DZ,DZA,12,ISO 3166-2:DZ,Africa,Northern Africa,NaN,2.0,15.0,NaN
4,American Samoa,AS,ASM,16,ISO 3166-2:AS,Oceania,Polynesia,NaN,9.0,61.0,NaN


(249, 11)

In [36]:
# Use .info() to inspect the data storage types and non-missing values.
# Count missing values in every feature.
outages_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 27 columns):
 #   Column                                  Non-Null Count  Dtype              
---  ------                                  --------------  -----              
 0   outage_id                               140 non-null    int64              
 1   Annotations data Source                 140 non-null    object             
 2   description                             137 non-null    object             
 3   Annotations scope                       0 non-null      float64            
 4   start_date                              140 non-null    datetime64[ns, UTC]
 5   end_date                                139 non-null    datetime64[ns, UTC]
 6   location_code                           111 non-null    object             
 7   Annotations asns                        90 non-null     object             
 8   Annotations geo Ids                     28 non-null     object             
 9  

In [37]:
# Count missing values in every feature of the outage table.
outages_df.isnull().sum()


outage_id                                   0
Annotations data Source                     0
description                                 3
Annotations scope                         140
start_date                                  0
end_date                                    1
location_code                              29
Annotations asns                           50
Annotations geo Ids                       112
Annotations origins                       133
Annotations entities.entity Type            0
Annotations entities.entity Value           0
Annotations entities.entity Name            0
entity_location_codes                      29
Annotations event Type                      0
Annotations linked Url                     19
Annotations origins Details.name          133
Annotations origins Details.origin        133
outage_cause                                0
outage_type                                 0
Annotations locations Details.name         29
Annotations locations Details.code

In [38]:
# Check the coverage: the dates, the outage types and causes, and the country-code lookup.
print("Earliest outage start:", outages_df["start_date"].min())
print("Latest outage start:", outages_df["start_date"].max())
print("Outage identifiers are unique:", outages_df["outage_id"].is_unique)
print("Different location codes in the file:", outages_df["location_code"].nunique())

display(outages_df["outage_type"].value_counts())
display(outages_df["outage_cause"].value_counts())

print("Rows in the country-code lookup:", len(country_codes_df))
print("Two-letter codes in the lookup are unique:", country_codes_df["alpha-2"].is_unique)
print("Missing codes or names in the lookup:", country_codes_df[["alpha-2", "alpha-3", "name"]].isnull().sum().sum())
display(country_codes_df[["alpha-2", "alpha-3", "name"]].dtypes)


Earliest outage start: 2025-11-18 09:15:00+00:00
Latest outage start: 2026-11-12 16:17:00+00:00
Outage identifiers are unique: True
Different location codes in the file: 49


outage_type
REGIONAL      50
NATIONWIDE    47
NETWORK       35
PLATFORM       8
Name: count, dtype: int64

outage_cause
GOVERNMENT_DIRECTED    40
POWER_OUTAGE           35
MILITARY_ACTION        14
CABLE_CUT              10
TECHNICAL_PROBLEM       9
UNKNOWN                 8
NATURAL_DISASTER        8
WEATHER                 6
CYBERATTACK             4
NETWORK_PROBLEM         2
MAINTENANCE             1
MECHANICAL              1
DNS                     1
SOFTWARE                1
Name: count, dtype: int64

Rows in the country-code lookup: 249
Two-letter codes in the lookup are unique: True
Missing codes or names in the lookup: 0


alpha-2    object
alpha-3    object
name       object
dtype: object

**Complete the table for every feature used in the question, preparation, or figures. Add rows as needed.**

| Feature | Meaning and units or codes | Feature type | Data storage type | Missing values |
| --- | --- | --- | --- | --- |
| `outage_id` | Cloudflare's identifier for the outage record (source column `Annotations id`) | categorical | int64 | 0 |
| `start_date` | Date and time the outage started, in UTC | temporal | datetime64[ns, UTC] | 0 |
| `end_date` | Date and time the outage ended, in UTC | temporal | datetime64[ns, UTC] | 1 |
| `location_code` | ISO 3166-1 two-letter code of the affected country | categorical | object | 29 |
| `entity_location_codes` | Two-letter country codes of the networks or regions named in the record, separated by semicolons | categorical | object | 29 |
| `outage_type` | Scope of the outage: NATIONWIDE, REGIONAL, NETWORK, or PLATFORM | categorical | object | 0 |
| `outage_cause` | Cause assigned by Cloudflare, such as GOVERNMENT_DIRECTED, POWER_OUTAGE, or CABLE_CUT | categorical | object | 0 |
| `description` | Cloudflare's short text description of the outage | categorical | object | 3 |
| `alpha-2` (lookup) | ISO 3166-1 two-letter country code | categorical | object | 0 |
| `alpha-3` (lookup) | ISO 3166-1 three-letter country code | categorical | object | 0 |
| `name` (lookup) | Country or territory name | categorical | object | 0 |
| `country_code` (created) | Country used for the outage: `location_code`, or the country of the affected network or region when that is empty | categorical | object | 0 |
| `month` (created) | Year and month the outage started, such as 2026-04 | temporal | object | 0 |
| `duration_hours` (created) | Hours from `start_date` to `end_date`, used only in the check | quantitative | float64 | 0 |

Use quantitative, categorical, or temporal for feature type.

Your answer:

The three created features are counted in `located_df`, the 132 records I kept. The other counts are for the full files. `outage_id` is stored as a number, but it is a label, so I list it as categorical.


**Which observations and features will you use? Explain your time periods, geographic grouping, filtering, missing-value decisions, and any counts, means, sums, or rates.**

Your answer:

**Observations.** I use the 132 outages that can be tied to a country. The 8 platform outages are left out because Cloudflare records them against a cloud service or its regions, not against a country. The Microsoft 365 record, for example, lists 80 Microsoft regions around the world.

**Country.** I use the record's own country code when it has one, which is 111 records. For the 21 network and regional records where that field is empty, I use the country listed for the affected network or region. I checked that none of those 21 names more than one country. When reading the file I kept the text NA as a value, because NA is Namibia's country code and Pandas would otherwise read it as missing.

**Time.** I group by the month in which the outage started, in UTC. That gives 13 calendar months, from November 2025 to November 2026. The first and last are partial, because the first outage starts on November 18, 2025 and I downloaded the file on November 15, 2026.

**Missing values.** One record has no `end_date` and three have no description. That does not affect the figures, because I count outages by when they start. No record is missing a start date, a type, or a cause.

**Summaries.** Every value is a count of outage records. For the map I count the outages in each country over the whole period. For the bar chart I count the outages in each month, split into the five countries with the most outages and one group for all other countries. A zero in that table means the source lists no outage for that group in that month. It is a count of zero records, not a missing value filled in.


In [39]:
# Prepare the analysis data and summaries. Sort time values within each relevant group.
# Report the original and retained row counts and any unmatched geographic identifiers.
# Platform outages are recorded against a cloud service or its regions, not a country, so they are set aside.
platform_df = outages_df.loc[outages_df["outage_type"] == "PLATFORM"]
located_df = outages_df.loc[outages_df["outage_type"] != "PLATFORM"].copy()

# Use the record's own country code. Where it is empty, use the country listed for the affected network or region.
located_df["country_code"] = located_df["location_code"]
needs_fill = located_df["country_code"].isna()
fill_codes = located_df.loc[needs_fill, "entity_location_codes"].str.split(";")
print("Filled records whose networks or regions name more than one country:", (fill_codes.apply(set).str.len() > 1).sum())
located_df.loc[needs_fill, "country_code"] = fill_codes.str[0]

print("Records in outages_df:", len(outages_df))
print("Platform records removed:", len(platform_df))
print("Records kept in located_df:", len(located_df))
print("Kept records that use their own country code:", (~needs_fill).sum())
print("Kept records filled from the affected network or region:", needs_fill.sum())
print("Kept records still without a country code:", located_df["country_code"].isna().sum())

# Label each outage with the month it started, then attach the three-letter code and country name.
located_df["month"] = located_df["start_date"].dt.strftime("%Y-%m")
located_df = located_df.merge(
    country_codes_df[["alpha-2", "alpha-3", "name"]],
    left_on="country_code",
    right_on="alpha-2",
    how="left"
).rename(columns={"alpha-3": "iso_alpha3", "name": "country"})

# Put the records in time order.
located_df = located_df.sort_values("start_date").reset_index(drop=True)

print("Country codes with no match in the lookup:", located_df["iso_alpha3"].isna().sum())
print("Countries with at least one outage:", located_df["country_code"].nunique())
print("First outage start in located_df:", located_df["start_date"].min())
print("Last outage start in located_df:", located_df["start_date"].max())


Filled records whose networks or regions name more than one country: 0
Records in outages_df: 140
Platform records removed: 8
Records kept in located_df: 132
Kept records that use their own country code: 111
Kept records filled from the affected network or region: 21
Kept records still without a country code: 0
Country codes with no match in the lookup: 0
Countries with at least one outage: 53
First outage start in located_df: 2025-11-18 09:15:00+00:00
Last outage start in located_df: 2026-11-08 03:15:00+00:00


In [40]:
# Display the platform records that were removed.
# Count the cloud regions listed on each platform record.
platform_regions = platform_df["Annotations origins"].str.split(";").str.len()
print("Most cloud regions listed on one platform record:", int(platform_regions.max()))

platform_df.assign(regions_listed=platform_regions)[["outage_id", "start_date", "outage_cause", "location_code", "regions_listed", "description"]]


Most cloud regions listed on one platform record: 80


,outage_id,start_date,outage_cause,location_code,regions_listed,description
0,1749,2026-11-12 16:17:00+00:00,TECHNICAL_PROBLEM,NaN,80.0,A global Microsoft 365 disruption impacted Exchange Onli...
16,1727,2026-10-20 05:45:00+00:00,TECHNICAL_PROBLEM,NaN,1.0,A technical problem at Google Cloud caused elevated erro...
53,1659,2026-09-01 14:30:00+00:00,NETWORK_PROBLEM,NaN,1.0,Multiple products in us-central1-b are experiencing netw...
85,1621,2026-06-23 21:00:00+00:00,UNKNOWN,NaN,NaN,Meta outage affecting social media and messaging platforms
104,1541,2026-03-31 08:00:00+00:00,MILITARY_ACTION,NaN,1.0,Iranian strikes on AWS data centers
107,1538,2026-03-23 22:30:00+00:00,MILITARY_ACTION,NaN,1.0,Disruption due to drone activity
112,1533,2026-03-02 05:00:00+00:00,POWER_OUTAGE,NaN,1.0,Localized power issue affecting APIs and instances in a ...
113,1532,2026-03-01 12:00:00+00:00,POWER_OUTAGE,NaN,1.0,Connectivity and power issues affecting APIs and instanc...


In [41]:
# Display the records whose country was filled from the affected network or region, and count them by type.
filled_df = outages_df.loc[(outages_df["outage_type"] != "PLATFORM") & outages_df["location_code"].isna()]
display(filled_df["outage_type"].value_counts())
filled_df[["outage_id", "start_date", "outage_type", "location_code", "entity_location_codes"]]


outage_type
NETWORK     15
REGIONAL     6
Name: count, dtype: int64

,outage_id,start_date,outage_type,location_code,entity_location_codes
4,1744,2026-11-03 17:30:00+00:00,REGIONAL,NaN,UA
7,1739,2026-10-27 17:15:00+00:00,NETWORK,NaN,HT
9,1734,2026-10-23 21:45:00+00:00,REGIONAL,NaN,UA;UA
10,1733,2026-10-22 10:15:00+00:00,REGIONAL,NaN,UA
11,1732,2026-10-21 22:30:00+00:00,NETWORK,NaN,US
12,1731,2026-10-21 09:30:00+00:00,NETWORK,NaN,FR
13,1743,2026-10-20 23:00:00+00:00,NETWORK,NaN,RU
14,1730,2026-10-20 20:00:00+00:00,NETWORK,NaN,RU
15,1729,2026-10-20 07:00:00+00:00,NETWORK,NaN,RU
17,1728,2026-10-19 22:00:00+00:00,NETWORK,NaN,RU


In [42]:
# Count the outages for each country: one row per country.
country_totals_df = (
    located_df.groupby(["iso_alpha3", "country"])
    .agg(outages=("outage_id", "count"), first_month=("month", "min"), last_month=("month", "max"))
    .reset_index()
    .sort_values(["outages", "country"], ascending=[False, True])
    .reset_index(drop=True)
)
country_totals_df["share_of_outages_percent"] = (country_totals_df["outages"] / country_totals_df["outages"].sum() * 100).round(1)
display(country_totals_df.head(12))

print("Countries in country_totals_df:", len(country_totals_df))
print("Outages counted:", country_totals_df["outages"].sum())
print(f"Share of outages in the five countries with the most: {country_totals_df['outages'].head(5).sum() / country_totals_df['outages'].sum() * 100:.1f}%")
print("Countries with exactly one outage:", (country_totals_df["outages"] == 1).sum())


,iso_alpha3,country,outages,first_month,last_month,share_of_outages_percent
0,IRQ,Iraq,26,2026-04,2026-10,19.7
1,SDN,Sudan,10,2026-04,2026-04,7.6
2,UKR,Ukraine,10,2025-12,2026-11,7.6
3,RUS,Russian Federation,8,2026-04,2026-10,6.1
4,CUB,Cuba,7,2026-03,2026-09,5.3
5,HTI,Haiti,4,2025-11,2026-10,3.0
6,PRT,Portugal,4,2026-01,2026-09,3.0
7,DOM,Dominican Republic,3,2025-12,2026-11,2.3
8,GUM,Guam,3,2026-04,2026-10,2.3
9,IRN,"Iran, Islamic Republic of",3,2026-01,2026-09,2.3


Countries in country_totals_df: 53
Outages counted: 132
Share of outages in the five countries with the most: 46.2%
Countries with exactly one outage: 33


In [43]:
# Count the outages for each month and country group: one row per month and group, with zeros kept.
top_countries = country_totals_df["country"].head(5).tolist()
country_groups = top_countries + ["All other countries"]
located_df["country_group"] = located_df["country"].where(located_df["country"].isin(top_countries), "All other countries")

# List every month from the first outage to the last, so a month with no outages would still appear.
all_months = pd.period_range("2025-11", "2026-11", freq="M").strftime("%Y-%m").tolist()

month_group_grid = (
    located_df.groupby(["month", "country_group"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=all_months, columns=country_groups, fill_value=0)
)
month_group_grid["All countries"] = month_group_grid.sum(axis=1)
display(month_group_grid)

month_group_df = (
    month_group_grid[country_groups]
    .stack()
    .rename("outages")
    .reset_index()
)
print("Rows in month_group_df:", len(month_group_df), "=", len(all_months), "months x", len(country_groups), "groups")
print("Outages counted:", month_group_df["outages"].sum())


country_group,Iraq,Sudan,Ukraine,Russian Federation,Cuba,All other countries,All countries
month,,,,,,,
2025-11,0,0,0,0,0,4,4
2025-12,0,0,2,0,0,5,7
2026-01,0,0,2,0,0,9,11
2026-02,0,0,0,0,0,4,4
2026-03,0,0,0,0,3,3,6
2026-04,1,10,0,1,0,2,14
2026-05,0,0,0,0,0,1,1
2026-06,3,0,0,0,0,3,6
2026-07,3,0,1,0,2,6,12


Rows in month_group_df: 78 = 13 months x 6 groups
Outages counted: 132


**How many records were removed from the dataset? Explain why, and state what one row represents after any aggregation.**

Your answer:

8 of the 140 records were removed, which leaves 132. All 8 are platform outages. They include the Microsoft 365 disruption on November 12, 2026, a Google Cloud problem on October 20, 2026, and a Meta outage on June 23, 2026. A platform outage is recorded against a cloud service or its regions. The Microsoft 365 record lists 80 Microsoft regions around the world and no country code, so it cannot be placed in one country on the map or in one country group.

After aggregation, one row of `country_totals_df` is one country with its number of outages, and there are 53 rows. One row of `month_group_df` is one month and one country group with its number of outages. It has 78 rows, which is 13 months times 6 groups, with zeros kept.


### Step 3: Evidence

Create two different Plotly Express figure types that answer this same question. Together they must display the spatial and temporal patterns. Use the prepared data consistently and explain any difference in coverage between the figures.

**Which two chart types will you use, and what part of the question should each make easier to answer? Specify the time, location, and quantitative encodings.**

Your answer:

**Figure 1 is a choropleth map** made with `px.choropleth`. Location is the country, given by its three-letter code. Color is the number of outages recorded in the whole period, on one scale from 0 to 26. It should make the where part of the question easy to answer. Time appears only in the hover, as each country's first and last month.

**Figure 2 is a stacked bar chart** made with `px.bar`. Time is on the horizontal axis, as the month the outage started. The height of each bar is the number of outages in that month. Color is the country group: the five countries with the most outages, and all other countries. It should make the when part easy to answer, and the colors show which countries are behind each month. I used bars and not a line because each month's value is a count of separate events, not one measurement changing over time.

Both figures use the same 132 records. The map shows all 53 countries. The bar chart names five of them and puts the other 48 into one gray group.


In [44]:
# Create Figure 1 with a descriptive title, units, useful hover fields, and suitable dimensions.
# If animated, use chronological frames and fixed scales.
fig = px.choropleth(
    country_totals_df,
    locations="iso_alpha3",
    color="outages",
    color_continuous_scale="Viridis",
    range_color=[0, country_totals_df["outages"].max()],
    hover_name="country",
    hover_data={"iso_alpha3": False, "outages": True, "first_month": True, "last_month": True},
    labels={"outages": "Outages recorded", "first_month": "First month", "last_month": "Last month"},
    projection="natural earth",
    title="Internet Outages Recorded by Cloudflare Radar by Country, November 2025 to November 2026"
)
fig.update_geos(projection_rotation_lon=0, showcountries=True, countrycolor="darkgray")
fig.update_layout(height=550)
fig.show()


In [45]:
# Count the outages by cause, by cause within the five leading countries, and by whether the description mentions an exam.
cause_counts = located_df["outage_cause"].value_counts().to_frame("outages")
cause_counts["share_percent"] = (cause_counts["outages"] / cause_counts["outages"].sum() * 100).round(1)
display(cause_counts)

# Causes for the five countries with the most outages.
display(pd.crosstab(located_df.loc[located_df["country"].isin(top_countries), "country"], located_df["outage_cause"]))

# Outages whose description mentions an exam.
exam_records = located_df.loc[located_df["description"].str.contains("exam", case=False, na=False)]
display(exam_records.groupby(["country", "outage_cause"]).size())
print("Outages whose description mentions an exam:", len(exam_records), "of", len(located_df))


,outages,share_percent
outage_cause,,
GOVERNMENT_DIRECTED,40,30.3
POWER_OUTAGE,33,25.0
MILITARY_ACTION,12,9.1
CABLE_CUT,10,7.6
NATURAL_DISASTER,8,6.1
UNKNOWN,7,5.3
TECHNICAL_PROBLEM,7,5.3
WEATHER,6,4.5
CYBERATTACK,4,3.0


outage_cause,CABLE_CUT,CYBERATTACK,GOVERNMENT_DIRECTED,MILITARY_ACTION,POWER_OUTAGE
country,,,,,
Cuba,0,0,0,0,7
Iraq,1,0,25,0,0
Russian Federation,1,1,0,6,0
Sudan,0,0,10,0,0
Ukraine,0,0,0,6,4


country  outage_cause       
Iraq     GOVERNMENT_DIRECTED    25
Sudan    GOVERNMENT_DIRECTED    10
dtype: int64

Outages whose description mentions an exam: 35 of 132


**What does Figure 1 show? Describe a pattern and record at least one value with its place, time, and units or count.**

Your answer:

Figure 1 shows that the outages are concentrated. Iraq is the one bright country, with 26 outages between April 2026 and October 2026, which is 19.7% of the 132. Sudan and Ukraine follow with 10 each, the Russian Federation has 8, and Cuba has 7. Those five countries hold 46.2% of the outages. Most of the other colored countries are the darkest shade, because 33 of the 53 countries have exactly one outage.

The colored countries are spread across Africa, Asia, Europe, and the Americas. The United States has 3.

Government-directed shutdowns are the most common cause, with 40 of the 132 outages (30.3%). In Iraq, 25 of the 26 outages are government directed.


In [46]:
# Create Figure 2 using a different chart type to answer the same research question.
# Label any aggregation or transformation.
fig = px.bar(
    month_group_df,
    x="month",
    y="outages",
    color="country_group",
    category_orders={"month": all_months, "country_group": country_groups},
    color_discrete_map={
        top_countries[0]: "firebrick",
        top_countries[1]: "goldenrod",
        top_countries[2]: "seagreen",
        top_countries[3]: "steelblue",
        top_countries[4]: "mediumpurple",
        "All other countries": "lightgray"
    },
    labels={"month": "Month the outage started (UTC)", "outages": "Outages recorded (count)", "country_group": "Country"},
    title="Internet Outages Recorded by Cloudflare Radar by Month and Country, November 2025 to November 2026"
)
fig.update_xaxes(type="category")
fig.update_layout(height=500)
fig.show()


In [47]:
# Total the two busiest months and compare them with the whole period.
monthly_totals = month_group_grid["All countries"]
busiest = monthly_totals.sort_values(ascending=False).head(2)
display(busiest)

print(f"Outages in the two busiest months: {busiest.sum()} of {monthly_totals.sum()} = {busiest.sum() / monthly_totals.sum() * 100:.1f}%")
print("Month with the fewest outages:", monthly_totals.idxmin(), "with", monthly_totals.min())
print("Months in which Iraq has at least one outage:", (month_group_grid["Iraq"] > 0).sum(), "of", len(month_group_grid))
print("Months in which Ukraine has at least one outage:", (month_group_grid["Ukraine"] > 0).sum(), "of", len(month_group_grid))
print("Months in which All other countries is the largest part of the bar:", (month_group_grid[country_groups].idxmax(axis=1) == "All other countries").sum(), "of", len(month_group_grid))


month
2026-10    25
2026-09    22
Name: All countries, dtype: int64

Outages in the two busiest months: 47 of 132 = 35.6%
Month with the fewest outages: 2026-05 with 1
Months in which Iraq has at least one outage: 6 of 13
Months in which Ukraine has at least one outage: 6 of 13
Months in which All other countries is the largest part of the bar: 9 of 13


**What does Figure 2 add? Record at least one value with its place, time, and units or count, and compare its message with Figure 1.**

Your answer:

Figure 2 adds the timing. The outages are not even across the year. October 2026 has 25 and September 2026 has 22, and together those two months hold 47 of the 132 outages (35.6%). May 2026 has 1.

It also shows that the leading countries reached their totals in different ways. Sudan's 10 outages are all in April 2026. Iraq's 26 are in 6 of the 13 months, with 10 in September 2026 alone. The Russian Federation has 6 of its 8 in October 2026. Ukraine's 10 are spread over 6 months, with no more than 2 in any month.

Figure 1 gives Sudan and Ukraine the same color, because each has 10 outages. Figure 2 shows that they are different cases. Sudan's outages are one burst in a single month, and Ukraine's run from December 2025 to November 2026. The map cannot show that. The bar chart cannot show which countries are inside the gray group, and that group is the largest part of the bar in 9 of the 13 months.


### Step 4: Check

Use hover to compare at least two plotted values with the source records or a calculated summary. Check chronological order, location matching, and one unusual high, low, or apparent gap that matters to your question. Explain any problem you corrected. Do not repeat the initial missing-value count.

In [48]:
# Verify selected plotted values against records or summaries.
# Check the relevant time order, geographic match, or unusual observation.
# Check 1: count Iraq's records directly in the source file and compare with the value on the map.
print("Source records with location code IQ:", (outages_df["location_code"] == "IQ").sum())
print("Value plotted for Iraq on the map:", country_totals_df.loc[country_totals_df["country"] == "Iraq", "outages"].iloc[0])
print("Sum of Iraq's bar segments:", month_group_grid["Iraq"].sum())


Source records with location code IQ: 26
Value plotted for Iraq on the map: 26
Sum of Iraq's bar segments: 26


In [49]:
# Check 2: read the source records behind the tallest single-country bar segment outside Iraq.
sudan_records = outages_df.loc[
    outages_df["location_code"] == "SD",
    ["outage_id", "start_date", "end_date", "outage_type", "outage_cause", "description"]
].sort_values("start_date")
display(sudan_records)

print(sudan_records["description"].unique())
print("Source records with location code SD:", len(sudan_records))
print("Value plotted for Sudan in 2026-04:", month_group_grid.loc["2026-04", "Sudan"])


,outage_id,start_date,end_date,outage_type,outage_cause,description
101,1576,2026-04-13 11:45:00+00:00,2026-04-13 15:15:00+00:00,NATIONWIDE,GOVERNMENT_DIRECTED,Suspension of Internet service during Sudanese Certifica...
99,1577,2026-04-14 11:45:00+00:00,2026-04-14 15:15:00+00:00,NATIONWIDE,GOVERNMENT_DIRECTED,Suspension of Internet service during Sudanese Certifica...
98,1579,2026-04-15 11:45:00+00:00,2026-04-15 15:15:00+00:00,NATIONWIDE,GOVERNMENT_DIRECTED,Suspension of Internet service during Sudanese Certifica...
97,1580,2026-04-16 11:45:00+00:00,2026-04-16 15:15:00+00:00,NATIONWIDE,GOVERNMENT_DIRECTED,Suspension of Internet service during Sudanese Certifica...
96,1581,2026-04-18 11:45:00+00:00,2026-04-18 15:15:00+00:00,NATIONWIDE,GOVERNMENT_DIRECTED,Suspension of Internet service during Sudanese Certifica...
95,1582,2026-04-19 11:45:00+00:00,2026-04-19 15:15:00+00:00,NATIONWIDE,GOVERNMENT_DIRECTED,Suspension of Internet service during Sudanese Certifica...
94,1583,2026-04-20 11:45:00+00:00,2026-04-20 15:15:00+00:00,NATIONWIDE,GOVERNMENT_DIRECTED,Suspension of Internet service during Sudanese Certifica...
93,1584,2026-04-21 11:45:00+00:00,2026-04-21 15:15:00+00:00,NATIONWIDE,GOVERNMENT_DIRECTED,Suspension of Internet service during Sudanese Certifica...
92,1585,2026-04-22 11:45:00+00:00,2026-04-22 15:15:00+00:00,NATIONWIDE,GOVERNMENT_DIRECTED,Suspension of Internet service during Sudanese Certifica...
91,1586,2026-04-23 11:45:00+00:00,2026-04-23 15:15:00+00:00,NATIONWIDE,GOVERNMENT_DIRECTED,Suspension of Internet service during Sudanese Certifica...


['Suspension of Internet service during Sudanese Certificate Examination sessions']
Source records with location code SD: 10
Value plotted for Sudan in 2026-04: 10


In [50]:
# Check 3: time order and location matching, including Namibia's country code.
print("Months on the horizontal axis:", all_months)
print("Months are in time order:", all_months == sorted(all_months))
print("Every month from the first outage to the last has a row:", sorted(located_df["month"].unique()) == all_months)
print("Every mapped country code is in the lookup:", located_df["country_code"].isin(country_codes_df["alpha-2"]).all())

located_df.loc[located_df["country_code"] == "NA", ["outage_id", "start_date", "country_code", "iso_alpha3", "country", "outage_type", "outage_cause"]]


Months on the horizontal axis: ['2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09', '2026-10', '2026-11']
Months are in time order: True
Every month from the first outage to the last has a row: True
Every mapped country code is in the lookup: True


,outage_id,start_date,country_code,iso_alpha3,country,outage_type,outage_cause
110,1721,2026-10-14 23:30:00+00:00,NA,NAM,Namibia,NATIONWIDE,POWER_OUTAGE


In [51]:
# Check 4: the lowest month. Count its records in the source file, with the platform records included.
source_month = outages_df["start_date"].dt.strftime("%Y-%m")
print("Source records that start in 2026-05, platform records included:", (source_month == "2026-05").sum())
outages_df.loc[source_month == "2026-05", ["outage_id", "start_date", "location_code", "outage_type", "outage_cause", "description"]]


Source records that start in 2026-05, platform records included: 1


,outage_id,start_date,location_code,outage_type,outage_cause,description
89,1588,2026-05-05 19:30:00+00:00,DE,NATIONWIDE,DNS,Distribution of faulty DNSSEC signatures


In [52]:
# Compare how long single records last: Sudan's exam-day shutdowns and the longest records in located_df.
located_df["duration_hours"] = ((located_df["end_date"] - located_df["start_date"]).dt.total_seconds() / 3600).round(1)
print("Duration of each Sudan record in hours:", located_df.loc[located_df["country"] == "Sudan", "duration_hours"].unique())
print("Median duration of all records in located_df:", located_df["duration_hours"].median(), "hours")

located_df.sort_values("duration_hours", ascending=False)[["outage_id", "country", "start_date", "end_date", "outage_cause", "duration_hours"]].head(5)


Duration of each Sudan record in hours: [3.5]
Median duration of all records in located_df: 3.5 hours


,outage_id,country,start_date,end_date,outage_cause,duration_hours
25,1531,"Iran, Islamic Republic of",2026-02-28 07:00:00+00:00,2026-05-26 12:00:00+00:00,GOVERNMENT_DIRECTED,2093.0
14,1518,"Iran, Islamic Republic of",2026-01-08 16:30:00+00:00,2026-02-01 00:00:00+00:00,GOVERNMENT_DIRECTED,559.5
20,1524,Portugal,2026-01-28 04:10:36+00:00,2026-02-18 00:00:00+00:00,WEATHER,499.8
3,1507,Indonesia,2025-11-25 22:00:47+00:00,2025-12-11 21:00:00+00:00,WEATHER,383.0
1,1506,Sri Lanka,2025-11-25 02:00:11+00:00,2025-12-09 00:00:00+00:00,WEATHER,334.0


**Which two values did you check, and did they agree? What did your check establish about order, location, or an unusual pattern?**

Your answer:

**Value 1: Iraq on the map.** The hover for Iraq in Figure 1 shows 26 outages. Counting the rows with location code IQ directly in the source file also gives 26, and Iraq's segments in Figure 2 add up to 26. They agree.

**Value 2: Sudan in April 2026.** The hover for the gold segment of the April 2026 bar in Figure 2 shows 10. The source file has 10 records with location code SD. They start on ten different days from April 13 to April 23, 2026, each one from 11:45 to 15:15 UTC, and all ten are nationwide and government directed. The description on every one is "Suspension of Internet service during Sudanese Certificate Examination sessions". The plotted value agrees with the records. It also shows what one count means in this dataset: each exam-day shutdown is its own record.

**Order and location.** The 13 months are in time order from left to right, and every month from the first outage to the last has a bar. All 53 country codes matched the lookup. Namibia's record kept its code, NA, and is on the map as NAM.

**Unusual value.** May 2026 is the lowest bar, with 1 outage. The source file, with the platform records included, also has only 1 record that starts in May 2026, a DNS problem in Germany on May 5. The low bar is in the source and is not something my filtering caused.


### Step 5: Conclusion

Restate your research question, then answer it using evidence from **both** figures. Include a comparison across places and a comparison across times, with values and units. Explain whether the evidence supports your prediction.

**Restate your question and write your conclusion.**

Your answer:

**Question.** Between November 18, 2025 and November 12, 2026, which countries had the most Internet outages recorded by Cloudflare Radar, and in which months did those outages happen?

**Where.** Figure 1 shows the outages concentrated in a few countries. Iraq has 26, Sudan and Ukraine have 10 each, the Russian Federation has 8, and Cuba has 7. Those five hold 46.2% of the 132 outages that have a country, while 33 of the 53 countries have one outage each. Iraq has more than twice as many as any other country.

**When.** Figure 2 shows the outages bunched in time. September and October 2026 hold 47 of the 132 (35.6%), against 1 in May 2026 and 4 in February 2026. The bunching comes from particular countries: 10 outages in Sudan in April 2026, 10 in Iraq in September 2026, and 6 in the Russian Federation in October 2026.

**Prediction.** The evidence mostly supports it. The outages are concentrated, four of the five countries at the top are ones with government shutdowns or a war, and the United States has only 3. The timing is uneven and comes in runs of days, as I predicted. Ukraine's 10 are 6 from military action and 4 power outages, and 6 of the Russian Federation's 8 are from military action. What I did not expect is how much of the count is school exams. The descriptions of 35 of the 132 outages mention an exam: 25 in Iraq and all 10 in Sudan. The fifth country, Cuba, does not fit my prediction, because all 7 of its outages are power outages.


### Step 6: Limitation

Identify **two limitations of your analysis**. Then explain which aspects of the research question are easier or harder to answer from each figure.

**What are two limitations, and how do the strengths and weaknesses of your figures affect the answer?**

Your answer:

**Limitation 1: this is a curated list, and I am counting records.** Cloudflare lists notable outages that it sees in its own traffic and can generally corroborate, so small or unreported outages are missing. A record is also not a fixed amount of harm. Each of Sudan's 10 records lasts 3.5 hours, which is 35 hours in total. The longest single record is a government-directed shutdown in Iran that lasted 2,093 hours, from February 28 to May 26, 2026, and it counts as 1. A count ranks Sudan far above Iran, and a measure of hours would do the opposite. The counts are not adjusted for population either.

**Limitation 2: the country and the calendar are both rough.** Eight platform outages are in neither figure, including the Microsoft 365 disruption that led me to this dataset. That outage was worldwide, and the file records it against 80 Microsoft regions with no country code. My figures count outages that belong to one country, so they say nothing about outages of online services that reach many countries at once. Only 47 of the 132 mapped outages are nationwide. The other 85 are regional or single-network outages, and each one still colors its whole country on the map. For 21 records the country comes from the affected network or region and not from the record's own country field. The first and last months are partial, so the bars for November 2025 and November 2026 should not be compared with full months.

**What each figure makes easier or harder.** The map makes the where easy. Iraq stands out at once and the spread across continents is visible. It cannot show when. It gives Sudan and Ukraine the same color for very different patterns, large countries such as the Russian Federation draw the eye more than small ones, and island territories such as Guam, which has 3 outages, are too small to see at world scale. The bar chart makes the when easy. The September and October peak and Sudan's April burst are clear. It loses the geography, and it hides 48 countries inside one gray group.


## AI-use reflection

**Did you use AI? If so, describe one suggestion you checked against your source, code, or output and what you accepted, modified, or rejected. If not, state that you did not use it.**

Your answer:

Yes. I used an AI assistant (Claude) to help find the two datasets, write the code, and draft the written answers, and I checked the answers against the notebook output.

One suggestion I checked and rejected: in Investigation 2, the first plan was to drop every outage record that had no country code, which would have removed 29 records. I checked those 29 in the source file. 21 of them are network or regional outages that do have a country, listed with the affected network or region. The other 8 are platform outages, which are recorded against a cloud service and not a country. I rejected dropping all 29. I kept the 21 with the country of their network or region and removed only the 8 platform records. The cell that lists those 21 records is in the notebook so the decision can be checked.


## Submission checklist

- Include two distinct datasets, each with both usable spatial and temporal information, plus any lookup or boundary files.
- Document both sources, feature definitions, units, collection context, coverage, and download filters.
- Complete the feature table and every written response for both investigations.
- Ask one research question per dataset and complete all six workflow steps for each question.
- Include four finished figures: two different chart types for each question, with each pair addressing both space and time.
- Explain filtering, missing values, aggregations, row counts, and any changes in the unit of observation.
- Check plotted values, temporal order, and geographic interpretation against the data.
- Restate each question in its conclusion and support the answer with evidence from both figures.
- Identify two limitations per investigation and compare what the two displays reveal or obscure.
- Complete the AI-use reflection. Restart the kernel, run cells in order, and ensure interactive figures display correctly before submitting.